# VDLF-Net 消融实验（Mini-ImageNet）

将 Jupyter 工作目录设为 **`experiment_ablation`** 后运行；数据默认使用 `../experiment_2/data/mini-imagenet`。

**流程**：环境 → 全局参数 → 加载数据 → 循环「训练 + 测试」各组 `AblationSpec` → 汇总 CSV。

审稿常用的 **KL / 重建 / 门控 / 单尺度** 组合已预置在 `default_ablation_presets()` 中。

In [5]:
import sys
from pathlib import Path

# tqdm 在 notebook 里若报 IProgress：pip install ipywidgets 后重启 kernel（可选）
AB_ROOT = Path.cwd().resolve()
E2_ROOT = AB_ROOT.parent / "experiment_2"
assert (E2_ROOT / "experiment_framework.py").is_file(), f"未找到 experiment_2: {E2_ROOT}"

if str(E2_ROOT) not in sys.path:
    sys.path.insert(0, str(E2_ROOT))
if str(AB_ROOT) not in sys.path:
    sys.path.insert(0, str(AB_ROOT))

import csv
import torch
from experiment_framework import FewShotConfig, EpisodeSampler
from ablation_spec import AblationSpec, default_ablation_presets
from vdlf_ablation import VDLFNetAblation
from ablation_train import train_vdlf_ablation, eval_ablation_on_test
from mini_imagenet_data import MiniImageNetDataset, get_transforms

print("AB_ROOT:", AB_ROOT)
print("device:", "cuda" if torch.cuda.is_available() else "cpu")

AB_ROOT: C:\Users\15366\Desktop\Thesis\Variational_Deep_Learning_Fusion_Modeling_for_High_Dimensional_Visual_Data__Feature_Adaptive_Approximation_and_Few_Shot_Inference\experiment_ablation
device: cuda


## 1. 全局参数

- **`RUN_PROTOCOL`**：
  - `"quick"`：40 / 15，冒烟与粗看趋势（方差大，不能定稿）。
  - **`"medium"`**：**150 / 50** —— 与 Table 2 **同训练轮数**，测试减半，总时间大约比 full 省一截（主要省在测试与多次验证）。
  - `"full"`：**150 / 100**，与 `table2_few_shot_experiment.ipynb` 一致，用于论文终表。
- 想再快一点：下面 **§3** 里可只跑 `full` + 你最关心的几组（如 `fine_scale_only`、`coarse_scale_only`、`uniform_gate`），最后再开 `full` 协议跑齐。

In [6]:
# quick | medium | full —— 建议：定结论用 medium 或 full，勿只用 quick
RUN_PROTOCOL = "full"

N_WAY = 5
K_SHOT = 5
Q_QUERY = 15
IMAGE_SIZE = 84

if RUN_PROTOCOL == "quick":
    NUM_TRAIN_EPISODES = 40
    NUM_TEST_EPISODES = 15
    VAL_INTERVAL = 20
elif RUN_PROTOCOL == "medium":
    NUM_TRAIN_EPISODES = 150
    NUM_TEST_EPISODES = 50
    VAL_INTERVAL = 25
elif RUN_PROTOCOL == "full":
    NUM_TRAIN_EPISODES = 150
    NUM_TEST_EPISODES = 100
    VAL_INTERVAL = 25
else:
    raise ValueError("RUN_PROTOCOL 须为 quick | medium | full")

BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 5e-5
NUM_SAMPLES = 15
TEMPERATURE = 15.0
ALPHA = 0.01
SEED = 42

DATA_ROOT = E2_ROOT / "data" / "mini-imagenet"
CKPT_DIR = AB_ROOT / "ablation_checkpoints"
CKPT_DIR.mkdir(exist_ok=True)

print("RUN_PROTOCOL:", RUN_PROTOCOL, "| train/test ep:", NUM_TRAIN_EPISODES, NUM_TEST_EPISODES)

RUN_PROTOCOL: full | train/test ep: 150 100


## 2. 数据集与采样器

In [7]:
assert DATA_ROOT.exists(), f"数据不存在: {DATA_ROOT}\n请放置 Mini-ImageNet 或修改 DATA_ROOT"

train_dataset = MiniImageNetDataset(DATA_ROOT, "train", get_transforms("train", IMAGE_SIZE))
val_dataset = MiniImageNetDataset(DATA_ROOT, "val", get_transforms("val", IMAGE_SIZE))
test_dataset = MiniImageNetDataset(DATA_ROOT, "test", get_transforms("val", IMAGE_SIZE))

train_sampler = EpisodeSampler(train_dataset, N_WAY, K_SHOT, Q_QUERY, train_dataset.get_class_to_indices())
val_sampler = EpisodeSampler(val_dataset, N_WAY, K_SHOT, Q_QUERY, val_dataset.get_class_to_indices())
test_sampler = EpisodeSampler(test_dataset, N_WAY, K_SHOT, Q_QUERY, test_dataset.get_class_to_indices())

print("OK: samplers", N_WAY, K_SHOT, Q_QUERY)

OK: samplers 5 5 15


## 3. 运行消融

结果写入 `ablation_results_K{K_SHOT}shot.csv`。

- **跑齐**：`PRESETS = default_ablation_presets()`。
- **省时间**：在代码格里用 `keep = {...}` 只保留几组名字（见下一格注释）；或 `default_ablation_presets()[:4]` 只跑前 4 组。

In [8]:
def make_config():
    return FewShotConfig(
        n_way=N_WAY,
        k_shot=K_SHOT,
        q_query=Q_QUERY,
        num_train_episodes=NUM_TRAIN_EPISODES,
        num_test_episodes=NUM_TEST_EPISODES,
        batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        latent_dim=128,
        num_samples=NUM_SAMPLES,
        temperature=TEMPERATURE,
        alpha=ALPHA,
        seed=SEED,
    )

PRESETS = default_ablation_presets()
# 想先快扫「反常」几组时再取消注释（终稿建议跑齐并把上一格的 RUN_PROTOCOL 设为 full）
# _keep = {"full", "fine_scale_only", "coarse_scale_only", "uniform_gate", "no_var"}
# PRESETS = [p for p in PRESETS if p.name in _keep]

rows = []

for spec in PRESETS:
    cfg = make_config()
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    model = VDLFNetAblation(cfg, spec)
    _, train_time, ckpt = train_vdlf_ablation(
        model, train_sampler, val_sampler, cfg, spec,
        ckpt_dir=CKPT_DIR, val_interval=VAL_INTERVAL,
    )
    mean_acc, ci, _, test_time = eval_ablation_on_test(model, test_sampler, cfg, ckpt_path=ckpt)

    rows.append({
        "ablation": spec.name,
        "drop_kl": spec.drop_kl,
        "drop_recon": spec.drop_recon,
        "gating_mode": spec.gating_mode,
        "mean_acc": round(mean_acc * 100, 2),
        "ci95": round(ci * 100, 2),
        "train_sec": round(train_time, 1),
        "test_sec": round(test_time, 1),
        "ckpt": ckpt.name,
    })
    print("→", spec.name, mean_acc * 100, "±", ci * 100)

out_csv = AB_ROOT / f"ablation_results_{K_SHOT}shot.csv"
with open(out_csv, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)
print("保存:", out_csv)

Train[full]:   0%|          | 0/150 [00:00<?, ?ep/s]

Ep 25/150 | Val: 83.19% ± 1.11% → saved best_ablation_full.pth
Ep 50/150 | Val: 84.59% ± 1.19% → saved best_ablation_full.pth
Ep 75/150 | Val: 84.00% ± 1.32%
Ep 100/150 | Val: 84.48% ± 1.14%
Ep 125/150 | Val: 85.08% ± 1.14% → saved best_ablation_full.pth
Ep 150/150 | Val: 85.69% ± 1.30% → saved best_ablation_full.pth


Evaluating:   0%|          | 0/100 [00:00<?, ?ep/s]

→ full 86.12000238895416 ± 1.1493290729811738


Train[no_kl]:   0%|          | 0/150 [00:00<?, ?ep/s]

...

→ fine_scale_only 87.36000204086304 ± 1.1273618923112492


Train[coarse_scale_only]:   0%|          | 0/150 [00:00<?, ?ep/s]

Ep 25/150 | Val: 80.77% ± 1.21% → saved best_ablation_coarse_scale_only.pth
Ep 50/150 | Val: 82.89% ± 1.24% → saved best_ablation_coarse_scale_only.pth
Ep 75/150 | Val: 81.99% ± 1.42%
Ep 100/150 | Val: 83.39% ± 1.24% → saved best_ablation_coarse_scale_only.pth
Ep 125/150 | Val: 83.72% ± 1.18% → saved best_ablation_coarse_scale_only.pth
Ep 150/150 | Val: 84.71% ± 1.27% → saved best_ablation_coarse_scale_only.pth


Evaluating:   0%|          | 0/100 [00:00<?, ?ep/s]

→ coarse_scale_only 85.28000235557556 ± 1.191244420913443
保存: C:\Users\15366\Desktop\Thesis\Variational_Deep_Learning_Fusion_Modeling_for_High_Dimensional_Visual_Data__Feature_Adaptive_Approximation_and_Few_Shot_Inference\experiment_ablation\ablation_results_5shot.csv


## 4.（可选）扩展 baseline

在 `experiment_2/baselines.py` 中已有 Prototypical / MAML 等；安装 `learn2learn` 后可在单独 cell 中训练 MAML，并用同一 `test_sampler` 评估，将一行追加进 CSV 以便与全文 Table 2 对照。